# 🎧 Customer Success Crew

## What We're Building

A 4-agent customer success pipeline that processes support tickets:

| Agent | Role |
|-------|------|
| **Complaint Analyst** | Parse support tickets, classify severity and sentiment |
| **Churn Risk Agent** | Assess churn probability based on complaint history |
| **Resolution Writer** | Draft personalized response and resolution plan |
| **Account Health Summarizer** | Produce an overall account health report |

## Key CrewAI Concept: `output_file`

We'll use `output_file` on the final task to save the account health report
to a file automatically — useful for batch-processing many accounts.

## Stack
- **CrewAI** — multi-agent pipeline
- **Ollama** — local LLM

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(model="ollama/qwen3.5:9b", base_url="http://localhost:11434")
print(f"LLM ready: {llm.model}")

## Step 2 — Customer Support Data

Simulated support ticket history for one customer account.

In [ ]:
account_info = """
ACCOUNT: Acme Corp (Enterprise Plan — $2,400/mo)
- Customer since: 18 months
- Contract renewal: 45 days away
- Monthly usage: Declining (was 85% of quota, now 62%)
- Champion contact: Sarah Chen (VP Ops) — on PTO for 2 weeks
- Technical contact: James Park (Senior Engineer) — increasingly frustrated
"""

support_tickets = """
RECENT SUPPORT TICKETS (last 30 days):

TICKET #4521 (5 days ago) — Priority: HIGH
From: James Park
Subject: API timeouts during peak hours — THIRD TIME this month
Body: "Our dashboard keeps timing out between 9-11am EST. This is the third
ticket I've filed about this. Last time I was told it was 'fixed' but clearly
it wasn't. We're evaluating alternatives. Please escalate."

TICKET #4490 (12 days ago) — Priority: MEDIUM
From: James Park
Subject: Missing data in export — rows dropped silently
Body: "Exported our Q3 report and noticed 2,300 rows are missing compared to
the in-app view. No error message. We had to manually reconcile. This caused
a 4-hour delay in our board presentation."

TICKET #4455 (20 days ago) — Priority: LOW
From: Lisa Wong (Analyst)
Subject: Feature request — custom date ranges in reports
Body: "Can we get custom date range filters? Right now it's only last 7/30/90
days. We need fiscal quarter views. Thanks!"

TICKET #4401 (28 days ago) — Priority: HIGH
From: James Park
Subject: API timeouts again
Body: "Same issue as ticket #4350. API returns 504 during morning hours.
Attached logs. This is blocking our automated reporting pipeline."
"""

csat_scores = """
CSAT SURVEY RESPONSES:
- 6 months ago: 9/10 ("Love the product, works great")
- 3 months ago: 6/10 ("Good product but reliability issues")
- 1 month ago: 3/10 ("Frustrated. Considering switching.")
"""

print(f"Loaded: account info, {support_tickets.count('TICKET')} tickets, 3 CSAT scores")

## Step 3 — Create Agents

In [ ]:
complaint_analyst = Agent(
    role="Customer Complaint Analyst",
    goal="Parse support tickets, classify severity, detect sentiment trends",
    backstory=(
        "You analyze support tickets for B2B SaaS accounts. You read between "
        "the lines — phrases like 'evaluating alternatives' and 'third time' are "
        "red flags. You categorize issues by type (bug, feature request, data "
        "quality) and track escalation patterns."
    ),
    llm=llm,
    verbose=True,
)

churn_risk_agent = Agent(
    role="Churn Risk Assessor",
    goal="Predict churn probability and identify the key risk factors",
    backstory=(
        "You've seen hundreds of accounts churn. You know the warning signs: "
        "declining usage, repeated same-issue tickets, frustration from the "
        "technical buyer, champion going silent, contract renewal approaching. "
        "You assign a churn score 0-100 and explain your reasoning."
    ),
    llm=llm,
    verbose=True,
)

resolution_writer = Agent(
    role="Resolution & Response Writer",
    goal="Draft empathetic, actionable responses and internal escalation plans",
    backstory=(
        "You write customer responses for high-risk accounts. Your emails are "
        "empathetic but honest — you acknowledge the problem, explain what went "
        "wrong, and propose a concrete fix timeline. You also draft the internal "
        "escalation plan for the engineering and CS teams."
    ),
    llm=llm,
    verbose=True,
)

health_summarizer = Agent(
    role="Account Health Summarizer",
    goal="Produce a concise account health report with risk score and action plan",
    backstory=(
        "You write the account health reports that the VP of Customer Success "
        "reviews every Monday. Each report must fit on one page: health score, "
        "key issues, churn risk, and a 30-day action plan. You rank accounts "
        "by urgency."
    ),
    llm=llm,
    verbose=True,
)

print("4 agents created")

## Step 4 — Create Tasks

Notice: the final task uses `output_file` — CrewAI will automatically save
the task output to a file on disk.

In [ ]:
complaint_task = Task(
    description=f"""Analyze these support tickets for the account:

{account_info}

{support_tickets}

{csat_scores}

For each ticket, determine:
1. Issue type: Bug / Data Quality / Feature Request / Performance
2. Severity: Critical / High / Medium / Low
3. Sentiment: Angry / Frustrated / Neutral / Positive
4. Repeat issue? Yes/No
5. Escalation signals (quotes that indicate churn risk)

Then provide an overall trend analysis:
- Are issues getting worse or better?
- What's the dominant complaint theme?
- Is this customer escalating or disengaging?""",
    expected_output="Ticket-by-ticket analysis with severity, sentiment, and trend.",
    agent=complaint_analyst,
)

churn_task = Task(
    description=f"""Based on the complaint analysis above, assess churn risk:

Account context:
{account_info}

Provide:
1. CHURN SCORE: 0-100 (0 = perfectly happy, 100 = already gone)
2. Risk factors (ranked by importance)
3. Protective factors (what's keeping them?)
4. Tipping point: What event would trigger churn?
5. Comparison: How does this look vs. typical pre-churn accounts?""",
    expected_output="Churn risk score with ranked risk factors and tipping point.",
    agent=churn_risk_agent,
)

resolution_task = Task(
    description="""Draft two communications:

1. EXTERNAL: Email to James Park (the frustrated technical contact)
   - Acknowledge the repeat API timeout issues
   - Explain what you're doing differently this time
   - Provide a specific resolution timeline
   - Offer a goodwill gesture (consider: service credit, dedicated support)
   - Tone: empathetic, professional, no corporate fluff

2. INTERNAL ESCALATION: Memo to engineering + CS leadership
   - Account value at risk ($2,400/mo = $28,800/yr ARR)
   - Technical root cause hypothesis
   - Requested engineering priority
   - 48-hour action items""",
    expected_output="External customer email + internal escalation memo.",
    agent=resolution_writer,
)

health_task = Task(
    description="""Compile the final Account Health Report:

Format:
## ACCOUNT HEALTH REPORT — [Account Name]

### Health Score: [🔴/🟡/🟢] [X/100]

### Account Summary
(2-3 sentences)

### Key Issues
(Ranked bullet list)

### Churn Risk
Score: X/100
Primary driver: [one sentence]

### 30-Day Action Plan
| Week | Action | Owner | Status |
|------|--------|-------|--------|
| Week 1 | ... | ... | Not started |

### Renewal Forecast
(Likelihood of renewal, conditions needed)""",
    expected_output="One-page account health report with score and action plan.",
    agent=health_summarizer,
    output_file="acme_corp_health_report.md",
    markdown=True,
)

print("4 tasks created")
print("Note: final task will write output to 'acme_corp_health_report.md'")

## Step 5 — Run the Crew

In [ ]:
cs_crew = Crew(
    agents=[complaint_analyst, churn_risk_agent, resolution_writer, health_summarizer],
    tasks=[complaint_task, churn_task, resolution_task, health_task],
    process=Process.sequential,
    verbose=True,
)

print("Customer Success crew assembled!")
result = cs_crew.kickoff()
print("\n✅ Account health report complete!")

In [ ]:
print("📋 ACCOUNT HEALTH REPORT:")
print("=" * 60)
print(result.raw)

In [ ]:
# View individual task outputs
labels = ["Complaint Analysis", "Churn Risk Assessment", "Resolution Drafts", "Health Report"]
for label, task_out in zip(labels, result.tasks_output):
    print(f"\n{'=' * 60}")
    print(f"📌 {label}")
    print("=" * 60)
    print(task_out.raw[:800])
    if len(task_out.raw) > 800:
        print("... (truncated)")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **`output_file`** | Final task auto-saves to `acme_corp_health_report.md` |
| **Sentiment analysis** | Detecting frustration signals in ticket language |
| **Churn scoring** | 0-100 risk score with ranked risk/protective factors |
| **Dual communications** | External (empathetic) + Internal (actionable) drafts |

## 🔧 Extensions

- **CRM integration**: Pull real tickets from Zendesk/Intercom
- **Batch processing**: Use `kickoff_for_each()` to run across all at-risk accounts
- **Automated alerts**: Trigger Slack alerts when churn score > 70
- **Historical tracking**: Store reports and track health trends over time